# 01 · Interchange vs `openmmforcefields` (theme A)

The same SMIRNOFF force field is applied to the same molecule by the two export
routes a downstream tool can take, and the resulting OpenMM `System`s are
required to be numerically identical:

1. **Interchange (direct):** `ForceField.create_openmm_system(topology)`.
2. **`openmmforcefields`:** a `SMIRNOFFTemplateGenerator` registered on an
`openmm.app.ForceField`, then `createSystem` on a `Modeller` topology whose
extra particles were added by `addExtraParticles`.

**What this notebook does and does not prove.** These are *not* two independent
implementations. `SMIRNOFFTemplateGenerator.generate_residue_template` calls
`ForceField.create_openmm_system`, which on this toolkit **is Interchange** — so
both routes share Interchange's virtual-site math, and a bug *inside* that math
would appear in both and cancel here. What is genuinely exercised is the
`openmmforcefields`-specific step: `convert_system_to_ffxml` **discards
Interchange's baked-in nonbonded exceptions** and writes only per-particle
σ/ε/charge plus `coulomb14scale`/`lj14scale`; OpenMM's `app.ForceField` then
*regenerates* every exception from the bond graph, giving each virtual site the
exclusions of its **first frame atom** (`excludeWith` → `getParticle(0)`).
Interchange independently registers each vsite under `orientations[0]` and
clones that atom's exception set, so the two agree — this notebook confirms the
ffxml round-trip preserves parameters, charges, and the exception *table*. The
checks that validate Interchange's vsite math against an *independent* reference
live in notebook 02 (native OpenMM water) and notebook 03 (closed-form geometry,
first-principles exclusion sets, hand-summed energies).

We cover **all four SMIRNOFF virtual-site types** (each with distinct distance,
charge-increment, and Lennard-Jones parameters) crossed with **four charge
methods**. Three are specified *inside the OFFXML* so the force field's own
charge-assignment code runs (`LibraryCharges`, `ToolkitAM1BCC`,
`ChargeIncrementModel`); the fourth supplies preset per-molecule charges
(production NAGL) via `charge_from_molecules`. We then push on adversarial
variations — molecule reordering, intra-molecule atom remapping, mixed
vsite/non-vsite systems, a net-charged molecule, non-default distinct 1-4
scaling, and the split nonbonded-force layout (also under PME) — plus a
robustness section confirming that force-field settings which would violate
OpenMM's exception model are *rejected*, never silently mis-handled.

Interchange places all vsites at the *end* of the system; `openmmforcefields`
interleaves them per residue — so every comparison goes through
`get_permutation_indices`, which matches vsites by **parent-atom identity**.

In [1]:
import warnings

import numpy as np
from openff.toolkit import ForceField, Molecule, Topology

import _helpers as H
import pathways as P

warnings.filterwarnings("ignore")
np.set_printoptions(precision=6, suppress=True)

RESULTS = []
NAGL_MODEL = "openff-gnn-am1bcc-1.0.0.pt"   # production NAGL model (Sage 2.3-era)

## Test molecules and virtual-site definitions

One representative molecule per vsite type, each vsite given **unique** geometry,
charge-increment, and LJ parameters. `match="all_permutations"` intentionally
generates multiple vsites where symmetry allows, stressing multi-vsite-per-parent
bookkeeping.

In [2]:
VDW = [("[#6:1]", 3.4, 0.10), ("[#1:1]", 2.6, 0.02),
       ("[#7:1]", 3.3, 0.17), ("[#8:1]", 3.0, 0.21),
       ("[#17:1]", 3.4, 0.30)]

VSITE_XML = {
    "BondCharge": (
        '<VirtualSite smirks="[#6:1]-[#17:2]" type="BondCharge" name="EP"'
        ' match="all_permutations" distance="-0.30*angstrom"'
        ' charge_increment1="0.00*elementary_charge" charge_increment2="0.20*elementary_charge"'
        ' sigma="1.05*angstrom" epsilon="0.030*kilocalorie/mole"/>'),
    "MonovalentLonePair": (
        '<VirtualSite smirks="[#8:1]=[#6X3:2]-[#6:3]" type="MonovalentLonePair" name="LP"'
        ' match="all_permutations" distance="0.55*angstrom" inPlaneAngle="118*degree"'
        ' outOfPlaneAngle="0*degree"'
        ' charge_increment1="0.11*elementary_charge" charge_increment2="-0.06*elementary_charge"'
        ' charge_increment3="-0.05*elementary_charge"'
        ' sigma="1.22*angstrom" epsilon="0.041*kilocalorie/mole"/>'),
    "DivalentLonePair": (
        '<VirtualSite smirks="[#8X2:1]([#6:2])[#6:3]" type="DivalentLonePair" name="LP"'
        ' match="all_permutations" distance="0.48*angstrom" outOfPlaneAngle="52*degree"'
        ' charge_increment1="0.12*elementary_charge" charge_increment2="-0.05*elementary_charge"'
        ' charge_increment3="-0.07*elementary_charge"'
        ' sigma="1.35*angstrom" epsilon="0.052*kilocalorie/mole"/>'),
    "TrivalentLonePair": (
        '<VirtualSite smirks="[#6:2][#7X3:1]([#6:3])[#6:4]" type="TrivalentLonePair" name="LP"'
        ' match="once" distance="0.42*angstrom"'
        ' charge_increment1="0.14*elementary_charge" charge_increment2="-0.04*elementary_charge"'
        ' charge_increment3="-0.05*elementary_charge" charge_increment4="-0.05*elementary_charge"'
        ' sigma="1.48*angstrom" epsilon="0.063*kilocalorie/mole"/>'),
}

TEST_SMILES = {
    "BondCharge": "[Cl:1][C:2]([H:6])([H:7])[C:3]([H:8])([H:9])[C:4]([H:10])([H:11])[Cl:5]",
    "MonovalentLonePair": "[C:1]([H:4])([H:5])([H:6])[C:2](=[O:3])[C:7]([H:8])([H:9])[H:10]",
    "DivalentLonePair": "[C:1]([H:4])([H:5])([H:6])[O:2][C:3]([H:7])([H:8])[H:9]",
    "TrivalentLonePair": "[C:1]([H:5])([H:6])([H:7])[N:2]([C:3]([H:8])([H:9])[H:10])[C:4]([H:11])([H:12])[H:13]",
}


def make_molecule(smiles):
    mol = Molecule.from_mapped_smiles(smiles)
    mol.generate_conformers(n_conformers=1)
    return mol

## Charge methods

Three methods are declared *in the OFFXML*, so both pathways exercise the force
field's own charge assignment (the `SMIRNOFFTemplateGenerator` runs the same
Interchange charge code internally):

- **`library`** — `<LibraryCharges>` (zero base; net charge comes from the
charge-conserving vsite increments)
- **`am1bcc`** — `<ToolkitAM1BCC>` (AmberTools computes AM1-BCC)
- **`charge_increment`** — `<ChargeIncrementModel>` (formal-charge base plus a
bond charge increment)

The fourth, **`preset_nagl`**, assigns production NAGL charges to the molecule
and feeds them through `charge_from_molecules` (auto-detected by the generator) —
the one non-OFFXML route, kept as a single representative case.

In [3]:
LIBQ0 = [("[#6:1]", 0.0), ("[#1:1]", 0.0), ("[#7:1]", 0.0),
         ("[#8:1]", 0.0), ("[#17:1]", 0.0)]

CHARGE_METHODS = ["library", "am1bcc", "charge_increment", "preset_nagl"]


def charge_block_for(method):
    if method in ("library", "preset_nagl"):
        return P.library_charge_block(LIBQ0)   # preset_nagl overrides this via charge_from_molecules
    if method == "am1bcc":
        return '<ToolkitAM1BCC version="0.3"></ToolkitAM1BCC>'
    if method == "charge_increment":
        return ('<ChargeIncrementModel version="0.4" partial_charge_method="formal_charge"'
                ' number_of_conformers="0"><ChargeIncrement smirks="[#6:1]-[#17:2]"'
                ' charge_increment1="0.05*elementary_charge"'
                ' charge_increment2="-0.05*elementary_charge"/></ChargeIncrementModel>')
    raise ValueError(method)


def build_systems(smiles_or_mol, vsite_type, method):
    """Build (interchange_system, ommffs_system, positions, molecule)."""
    mol = make_molecule(smiles_or_mol) if isinstance(smiles_or_mol, str) else smiles_or_mol
    offxml = P.build_offxml(VDW, [], [VSITE_XML[vsite_type]], charge_block=charge_block_for(method))
    ff = ForceField(offxml)
    charge_from = None
    if method == "preset_nagl":
        mol.assign_partial_charges(NAGL_MODEL)
        charge_from = [mol]
    top = mol.to_topology()
    ic = P.via_interchange(ff, top, charge_from_molecules=charge_from)
    og, pos = P.via_openmmforcefields(mol, offxml, top.to_openmm(), mol.conformers[0].to_openmm())
    return ic, og, pos, mol

## Core comparison — 4 vsite types × 4 charge methods

For each combination we require identical particle counts, per-particle nonbonded
parameters, and energies/forces at the input geometry **and** after 150 fs of
dynamics (a perturbed but physical geometry).

In [4]:
def compare_pathways(vsite_type, method, *, verbose=False):
    ic, og, pos, mol = build_systems(TEST_SMILES[vsite_type], vsite_type, method)
    n_vs = H.count_vsites(ic)
    assert n_vs > 0 and n_vs == H.count_vsites(og), \
        f"{vsite_type}/{method}: vsite count {n_vs} vs {H.count_vsites(og)}"

    ndiff = H.diff_systems(og, ic, "OMMFFS", "IC")
    assert abs(H.total_charge(ic)) < 1e-6 and abs(H.total_charge(og)) < 1e-6, "net charge != 0"

    # The openmmforcefields path regenerates exceptions from the bond graph
    # rather than copying Interchange's; require the full exception *tables*
    # (chargeProd, sigma, epsilon; vsites matched by parent ordinals) to agree,
    # not merely the totals — a dropped/extra/mis-scaled exception can be nearly
    # energy-invisible at one geometry.
    exc_diffs = H.compare_exception_tables(og, ic)
    assert not exc_diffs, f"{vsite_type}/{method}: exception-table diffs {exc_diffs[:6]}"

    d0, f0 = H.compare_energies(f"{vsite_type}/{method} @conf", pos, og, ic,
                                recompute_vsites=True, verbose=verbose)
    perturbed = H.propagate_dynamics(pos, og, nsteps=150)
    d1, f1 = H.compare_energies(f"{vsite_type}/{method} @perturbed", perturbed, og, ic,
                                recompute_vsites=True, verbose=False)
    RESULTS.append({"notebook": "01", "vsite_type": vsite_type, "charge_model": method,
                    "n_particles": ic.getNumParticles(), "n_vsites": n_vs, "nb_param_diffs": ndiff,
                    "max_energy_delta_kcal": max(abs(d0.value_in_unit(H.ENERGY_UNIT)),
                                                 abs(d1.value_in_unit(H.ENERGY_UNIT))),
                    "max_force_reldev": max(f0, f1), "status": "PASS"})
    print(f"  [{vsite_type:18s} / {method:16s}] vsites={n_vs} nb_diffs={ndiff} "
          f"maxΔE={max(abs(d0.value_in_unit(H.ENERGY_UNIT)), abs(d1.value_in_unit(H.ENERGY_UNIT))):.1e} kcal/mol  PASS")


for vtype in ["BondCharge", "MonovalentLonePair", "DivalentLonePair", "TrivalentLonePair"]:
    print(f"\n=== {vtype} ===")
    for method in CHARGE_METHODS:
        compare_pathways(vtype, method)
print("\nAll 4 vsite types × 4 charge methods: PASS")


=== BondCharge ===


  no nonbonded differences between OMMFFS and IC
  [BondCharge         / library         ] vsites=2 nb_diffs=0 maxΔE=1.3e-15 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [BondCharge         / am1bcc          ] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS
  no nonbonded differences between OMMFFS and IC
  [BondCharge         / charge_increment] vsites=2 nb_diffs=0 maxΔE=5.3e-17 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [BondCharge         / preset_nagl     ] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS

=== MonovalentLonePair ===
  no nonbonded differences between OMMFFS and IC
  [MonovalentLonePair / library         ] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [MonovalentLonePair / am1bcc          ] vsites=2 nb_diffs=0 maxΔE=0.0e+00 kcal/mol  PASS
  no nonbonded differences between OMMFFS and IC
  [MonovalentLonePair / charge_increment] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [MonovalentLonePair / preset_nagl     ] vsites=2 nb_diffs=0 maxΔE=1.7e-15 kcal/mol  PASS

=== DivalentLonePair ===
  no nonbonded differences between OMMFFS and IC
  [DivalentLonePair   / library         ] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [DivalentLonePair   / am1bcc          ] vsites=2 nb_diffs=0 maxΔE=0.0e+00 kcal/mol  PASS
  no nonbonded differences between OMMFFS and IC
  [DivalentLonePair   / charge_increment] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [DivalentLonePair   / preset_nagl     ] vsites=2 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS

=== TrivalentLonePair ===
  no nonbonded differences between OMMFFS and IC
  [TrivalentLonePair  / library         ] vsites=1 nb_diffs=0 maxΔE=0.0e+00 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [TrivalentLonePair  / am1bcc          ] vsites=1 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS
  no nonbonded differences between OMMFFS and IC
  [TrivalentLonePair  / charge_increment] vsites=1 nb_diffs=0 maxΔE=0.0e+00 kcal/mol  PASS


  no nonbonded differences between OMMFFS and IC
  [TrivalentLonePair  / preset_nagl     ] vsites=1 nb_diffs=0 maxΔE=5.4e-14 kcal/mol  PASS

All 4 vsite types × 4 charge methods: PASS


## Net-charged molecule

A charged species stresses formal-charge handling alongside vsite charge
increments. Chloroacetate (net −1) carries a `BondCharge` sigma-hole site on its
C–Cl bond. We use the two charge methods that respect formal charge
(`ToolkitAM1BCC` and preset NAGL) and require the exported net charge to equal −1
in both pathways, with identical energies.

In [5]:
CHLOROACETATE = "[Cl:1][C:2]([H:3])([H:4])[C:5](=[O:6])[O-:7]"


def compare_charged(method):
    ic, og, pos, mol = build_systems(CHLOROACETATE, "BondCharge", method)
    net_ic, net_og = H.total_charge(ic), H.total_charge(og)
    assert abs(net_ic - (-1.0)) < 1e-5, f"IC net charge {net_ic} != -1"
    assert abs(net_og - (-1.0)) < 1e-5, f"OMMFFS net charge {net_og} != -1"
    d, f = H.compare_energies(f"chloroacetate/{method}", pos, og, ic,
                              recompute_vsites=True, verbose=False)
    print(f"  [chloroacetate(-1) / {method:12s}] net IC={net_ic:+.4f} OMMFFS={net_og:+.4f}  "
          f"ΔE={d.value_in_unit(H.ENERGY_UNIT):.1e} kcal/mol  PASS")
    RESULTS.append({"notebook": "01", "vsite_type": "BondCharge (anion)", "charge_model": method,
                    "n_particles": ic.getNumParticles(), "n_vsites": H.count_vsites(ic),
                    "net_charge": net_ic, "max_energy_delta_kcal": abs(d.value_in_unit(H.ENERGY_UNIT)),
                    "max_force_reldev": f, "status": "PASS"})


print("\nNet-charged molecule (chloroacetate anion):")
for method in ("am1bcc", "preset_nagl"):
    compare_charged(method)


Net-charged molecule (chloroacetate anion):


  [chloroacetate(-1) / am1bcc      ] net IC=-1.0000 OMMFFS=-1.0000  ΔE=0.0e+00 kcal/mol  PASS


  [chloroacetate(-1) / preset_nagl ] net IC=-1.0000 OMMFFS=-1.0000  ΔE=0.0e+00 kcal/mol  PASS


## Adversarial variation 1 — molecule reordering, atom remapping, mixed systems

Interchange keeps molecule atoms contiguous and appends all vsites at the end;
`openmmforcefields` interleaves per residue. We mix vsite-bearing and plain
molecules, shuffle their order, and — crucially — **remap the atom order within**
copies of a molecule (`Molecule.remap`), so the two copies are chemically
identical but atom-indexed differently. This is the classic way to trip up
vsite↔parent indexing.

In [6]:
def compare_multi(mols, offxml, label):
    ff = ForceField(offxml)
    top = Topology()
    for mol in mols:
        top.add_molecule(mol)
    ic = P.via_interchange(ff, top)

    from openmm import Vec3
    from openmm.app import Modeller
    import openmm.unit as _u
    modeller = Modeller(mols[0].to_topology().to_openmm(), mols[0].conformers[0].to_openmm())
    offset = 2.5
    for mol in mols[1:]:
        raw = mol.conformers[0].to_openmm().value_in_unit(_u.nanometer)
        shifted = [Vec3(x + offset, y, z) for x, y, z in raw] * _u.nanometer
        modeller.add(mol.to_topology().to_openmm(), shifted)
        offset += 2.5
    og, pos = P.via_openmmforcefields(mols, offxml, modeller.topology, modeller.positions)

    assert H.count_vsites(ic) == H.count_vsites(og) > 0
    d, f = H.compare_energies(label, pos, og, ic, recompute_vsites=True, verbose=False)
    print(f"  [{label}] particles={ic.getNumParticles()} vsites={H.count_vsites(ic)} "
          f"ΔE={d.value_in_unit(H.ENERGY_UNIT):.1e}  PASS")
    RESULTS.append({"notebook": "01", "vsite_type": "multi", "charge_model": label,
                    "n_particles": ic.getNumParticles(), "n_vsites": H.count_vsites(ic),
                    "max_energy_delta_kcal": abs(d.value_in_unit(H.ENERGY_UNIT)),
                    "max_force_reldev": f, "status": "PASS"})


def remapped_copy(mol):
    """A chemically identical copy with reversed atom ordering."""
    n = mol.n_atoms
    new = mol.remap({i: n - 1 - i for i in range(n)}, current_to_new=True)
    new.generate_conformers(n_conformers=1)
    return new


combo_offxml = P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"], VSITE_XML["DivalentLonePair"]])
bondmol = make_molecule(TEST_SMILES["BondCharge"])
ethermol = make_molecule(TEST_SMILES["DivalentLonePair"])
plain = make_molecule("[C:1]([H:2])([H:3])([H:4])[H:5]")

compare_multi([bondmol, ethermol, plain], combo_offxml, "order A (bond, ether, methane)")
compare_multi([plain, bondmol, ethermol], combo_offxml, "order B (methane, bond, ether)")
# two chemically identical BondCharge molecules, second with reversed atom order
b1 = make_molecule(TEST_SMILES["BondCharge"])
compare_multi([b1, remapped_copy(b1)], P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]]),
              "duplicate BondCharge (2nd atom-remapped)")

  [order A (bond, ether, methane)] particles=29 vsites=4 ΔE=-1.1e-13  PASS
  [order B (methane, bond, ether)] particles=29 vsites=4 ΔE=0.0e+00  PASS
  [duplicate BondCharge (2nd atom-remapped)] particles=26 vsites=4 ΔE=0.0e+00  PASS


## Adversarial variation 2 — split nonbonded forces (interchange #1498, verifying the fix)

Interchange #1498: virtual-site nonbonded exceptions were incorrectly *zeroed*
when electrostatics and vdW are exported as **separate** forces
(`combine_nonbonded_forces=False`). This suite runs on an Interchange build that
**includes the #1498 fix**, so here we *verify the fix is correct and complete*:
the split-force system's total energy must equal the combined-force system's for
every vsite type, and a no-vsite control must be unaffected.

In [7]:
import importlib.metadata as _md
from packaging.version import Version

_IC_VERSION = Version(_md.version("openff-interchange"))
assert _IC_VERSION > Version("0.5.3"), (
    f"interchange {_IC_VERSION} predates the #1498 fix; install the patched build "
    "(see notebook 00 / README).")


def split_vs_combined_gap(smiles, vsite_type, *, with_vsite=True):
    vsites = [VSITE_XML[vsite_type]] if with_vsite else []
    offxml = P.build_offxml(VDW, LIBQ0, vsites)
    ff = ForceField(offxml)
    mol = make_molecule(smiles)
    top = mol.to_topology()
    ic_c = ff.create_interchange(top).to_openmm_system(combine_nonbonded_forces=True)
    ic_s = ff.create_interchange(top).to_openmm_system(combine_nonbonded_forces=False)
    pos = mol.conformers[0].to_openmm()
    e_c, _ = H.compute_energy(ic_c, pos, recompute_vsites=True)
    e_s, _ = H.compute_energy(ic_s, pos, recompute_vsites=True)
    return abs((e_c["total"] - e_s["total"]).value_in_unit(H.ENERGY_UNIT)), ic_s.getNumForces()


ctrl_gap, _ = split_vs_combined_gap(TEST_SMILES["BondCharge"], "BondCharge", with_vsite=False)
print(f"no-vsite control split-vs-combined gap: {ctrl_gap:.2e} kcal/mol (must be ~0)")
assert ctrl_gap <= 1e-6

print(f"\ninterchange {_IC_VERSION}: verifying #1498 fix (split == combined with vsites):")
for vtype in ["BondCharge", "MonovalentLonePair", "DivalentLonePair", "TrivalentLonePair"]:
    gap, nforces = split_vs_combined_gap(TEST_SMILES[vtype], vtype)
    print(f"  [{vtype:18s}] split forces={nforces}  gap={gap:.2e} kcal/mol  PASS")
    assert gap <= 1e-4, f"{vtype}: split-force vsite energy gap {gap:.4f} — #1498 fix incomplete!"
    RESULTS.append({"notebook": "01", "vsite_type": f"{vtype} (split NB)", "charge_model": "library",
                    "max_energy_delta_kcal": gap, "status": "PASS"})

print("\n#1498 fix verified: split and combined nonbonded-force layouts agree for all vsite types.")

no-vsite control split-vs-combined gap: 0.00e+00 kcal/mol (must be ~0)

interchange 0.5.4.dev4+g7e742e906: verifying #1498 fix (split == combined with vsites):
  [BondCharge        ] split forces=8  gap=7.23e-08 kcal/mol  PASS


  [MonovalentLonePair] split forces=8  gap=0.00e+00 kcal/mol  PASS
  [DivalentLonePair  ] split forces=8  gap=0.00e+00 kcal/mol  PASS
  [TrivalentLonePair ] split forces=8  gap=0.00e+00 kcal/mol  PASS

#1498 fix verified: split and combined nonbonded-force layouts agree for all vsite types.


## Adversarial variation 3 — non-default *distinct* 1-4 scaling through the ffxml

`convert_system_to_ffxml` reconstructs 1-4 interactions from `coulomb14scale`
and `lj14scale` alone (it does not copy Interchange's exceptions). If those
attributes were dropped or a single value applied to both, virtual-site 1-4
exceptions in the `openmmforcefields` system would silently diverge from a
direct Interchange export. We use **different** electrostatic and vdW 1-4
scales, both non-default, and require the full exception tables (values) and
the energy to match — on a molecule whose vsite genuinely has 1-4 partners.

In [8]:
def compare_nondefault_scaling(elec14, vdw14):
    mol = make_molecule(TEST_SMILES["BondCharge"])
    offxml = P.build_offxml(VDW, [], [VSITE_XML["BondCharge"]],
                            charge_block='<ToolkitAM1BCC version="0.3"></ToolkitAM1BCC>',
                            elec_scale14=elec14, vdw_scale14=vdw14)
    ff = ForceField(offxml)
    ic = P.via_interchange(ff, mol.to_topology())
    og, pos = P.via_openmmforcefields(mol, offxml, mol.to_topology().to_openmm(),
                                      mol.conformers[0].to_openmm())
    exc_diffs = H.compare_exception_tables(og, ic)
    d, f = H.compare_energies(f"scale elec={elec14} vdw={vdw14}", pos, og, ic,
                              recompute_vsites=True, verbose=False)
    print(f"  [elec14={elec14:<8} vdw14={vdw14:<8}] exception-value diffs={len(exc_diffs)}  "
          f"ΔE={d.value_in_unit(H.ENERGY_UNIT):.1e} kcal/mol  PASS")
    assert not exc_diffs, f"non-default scaling exception diffs: {exc_diffs[:6]}"
    RESULTS.append({"notebook": "01", "vsite_type": "BondCharge (non-default 1-4)",
                    "charge_model": f"elec14={elec14},vdw14={vdw14}",
                    "exception_table_diffs": len(exc_diffs),
                    "max_energy_delta_kcal": abs(d.value_in_unit(H.ENERGY_UNIT)), "status": "PASS"})


print("\nNon-default, distinct electrostatic/vdW 1-4 scaling round-trip:")
for e14, v14 in [(0.5, 0.75), (0.6, 0.4), (1.0, 0.0)]:
    compare_nondefault_scaling(e14, v14)


Non-default, distinct electrostatic/vdW 1-4 scaling round-trip:
  [elec14=0.5      vdw14=0.75    ] exception-value diffs=0  ΔE=0.0e+00 kcal/mol  PASS


  [elec14=0.6      vdw14=0.4     ] exception-value diffs=0  ΔE=5.4e-14 kcal/mol  PASS
  [elec14=1.0      vdw14=0.0     ] exception-value diffs=0  ΔE=0.0e+00 kcal/mol  PASS


## Adversarial variation 4 — split forces under PME with virtual-site 1-4 pairs

The OpenFE-relevant combination the single-molecule NoCutoff check above does
not cover: **periodic PME × `combine_nonbonded_forces=False` × a virtual site
with genuine scaled 1-4 partners** (water, being fully intra-excluded, never
exercises this). We pack a small periodic box of a vsite-bearing molecule and
require the split-force total energy and per-particle forces to match the
trusted combined-force layout across several geometries.

In [9]:
def pme_split_vs_combined(n_copies=6):
    import openmm
    import openmm.unit as _u
    from openff.units import unit as _ou
    from openff.interchange.components._packmol import pack_box, UNIT_CUBE

    mol = Molecule.from_smiles("ClCCCCl")
    top = pack_box(molecules=[mol], number_of_copies=[n_copies],
                   target_density=700 * _ou.kilogram / _ou.meter ** 3, box_shape=UNIT_CUBE)
    box = np.asarray(top.box_vectors.m_as(_ou.nanometer))
    pos = np.asarray(top.get_positions().m_as(_ou.nanometer))
    ff = ForceField(P.build_offxml(VDW, [], [VSITE_XML["BondCharge"]],
                                   charge_block='<ToolkitAM1BCC version="0.3"></ToolkitAM1BCC>'))
    ic = ff.create_interchange(top)
    combined = ic.to_openmm_system(combine_nonbonded_forces=True)
    split = ic.to_openmm_system(combine_nonbonded_forces=False)
    cutoff = float(np.min(np.diag(box))) / 2 - 0.05
    for s in (combined, split):
        for fo in s.getForces():
            if isinstance(fo, openmm.NonbondedForce):
                fo.setNonbondedMethod(openmm.NonbondedForce.PME)
                fo.setCutoffDistance(cutoff * _u.nanometer)
                fo.setEwaldErrorTolerance(1e-6)
                fo.setUseSwitchingFunction(False)
            elif isinstance(fo, openmm.CustomNonbondedForce):
                fo.setNonbondedMethod(openmm.CustomNonbondedForce.CutoffPeriodic)
                fo.setCutoffDistance(cutoff * _u.nanometer)
                fo.setUseSwitchingFunction(False)
        s.setDefaultPeriodicBoxVectors(*(box * _u.nanometer))
    arr = np.asarray(H._pad_positions(pos * _u.nanometer,
                                      combined.getNumParticles()).value_in_unit(_u.nanometer))
    rng = np.random.default_rng(1)
    worst_e = worst_f = 0.0
    for k in range(3):
        p = (arr if k == 0 else arr + rng.normal(0, 0.01, arr.shape)) * _u.nanometer
        ec, fc = H.compute_energy(combined, p, recompute_vsites=True, platform="CPU")
        es, fs = H.compute_energy(split, p, recompute_vsites=True, platform="CPU")
        rel = abs((ec["total"] - es["total"]).value_in_unit(H.ENERGY_UNIT)) / \
            max(abs(ec["total"].value_in_unit(H.ENERGY_UNIT)), 1.0)
        df = float(np.max(np.abs(fc["total"].value_in_unit(H.FORCE_UNIT)
                                 - fs["total"].value_in_unit(H.FORCE_UNIT))))
        worst_e, worst_f = max(worst_e, rel), max(worst_f, df)
    print(f"  {n_copies}x ClCCCCl, PME box: particles={combined.getNumParticles()} "
          f"vsites={H.count_vsites(combined)}  max rel ΔE={worst_e:.1e}  max ΔF={worst_f:.1e} kcal/mol/A")
    assert worst_e < 1e-5 and worst_f < 1e-2, "PME split/combined drift"
    RESULTS.append({"notebook": "01", "vsite_type": "BondCharge (PME split box)",
                    "charge_model": "am1bcc", "rel_energy_delta": worst_e,
                    "max_force_delta": worst_f, "status": "PASS"})


print("\nPeriodic PME split-vs-combined nonbonded forces (vsite 1-4 pairs):")
pme_split_vs_combined()


Periodic PME split-vs-combined nonbonded forces (vsite 1-4 pairs):


  6x ClCCCCl, PME box: particles=78 vsites=12  max rel ΔE=8.4e-08  max ΔF=1.2e-03 kcal/mol/A


## Robustness — settings that violate OpenMM's exception model are *rejected*

OpenMM's nonbonded exception machinery is hardwired: 1-2 and 1-3 pairs are
fully excluded, 1-5 pairs are full-strength, and each virtual site shares one
atom's exclusions. A SMIRNOFF force field can only *express* 1-4 scaling
(`scale14`) and the `"parents"` vsite policy within that model. We confirm that
every out-of-model setting raises **before** any system is built — there is no
silent path to a wrong energy:

- non-zero `scale12`, non-zero `scale13`, or non-unity `scale15` (electrostatics
*or* vdW) → rejected by the **OpenFF Toolkit** at force-field parse time;
- `exclusion_policy` other than `"parents"` → rejected by **Interchange** at
export (and independently by `openmmforcefields`);
- while any `scale14` is accepted and applied consistently by both paths.

In [10]:
def _rejected(build_ff):
    try:
        ff = build_ff()
    except Exception as e:
        return f"toolkit parse: {type(e).__name__}"
    mol = make_molecule(TEST_SMILES["BondCharge"])
    try:
        ff.create_interchange(mol.to_topology()).to_openmm_system()
    except Exception as e:
        return f"interchange export: {type(e).__name__}"
    return "BUILT (no error)"


guard_cases = {
    "elec scale13=0.5": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], elec_scale13=0.5)),
    "elec scale12=0.5": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], elec_scale12=0.5)),
    "elec scale15=0.5": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], elec_scale15=0.5)),
    "vdw  scale13=0.5": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], vdw_scale13=0.5)),
    "vdw  scale15=0.5": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], vdw_scale15=0.5)),
    "exclusion_policy=none": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], exclusion_policy="none")),
    "exclusion_policy=minimal": lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]], exclusion_policy="minimal")),
}
print("\nOut-of-model force-field settings must be rejected, not silently mis-handled:")
for label, builder in guard_cases.items():
    outcome = _rejected(builder)
    print(f"  {label:26s} -> {outcome}")
    assert "error" not in outcome.lower() or True  # informational label
    assert outcome != "BUILT (no error)", f"{label} was NOT rejected — silent-danger path!"
# and a valid non-default scale14 must build fine
assert _rejected(lambda: ForceField(P.build_offxml(VDW, LIBQ0, [VSITE_XML["BondCharge"]],
                                                   elec_scale14=0.7, vdw_scale14=0.3))) == "BUILT (no error)"
print("  scale14=0.7/0.3 (in-model)  -> BUILT (accepted, applied consistently)")
RESULTS.append({"notebook": "01", "vsite_type": "robustness guards",
                "charge_model": "scale12/13/15 + exclusion_policy", "status": "PASS"})
print("\nNo silent-danger path: every out-of-model setting raises at parse or export.")


Out-of-model force-field settings must be rejected, not silently mis-handled:
  elec scale13=0.5           -> toolkit parse: SMIRNOFFSpecError
  elec scale12=0.5           -> toolkit parse: SMIRNOFFSpecError
  elec scale15=0.5           -> toolkit parse: SMIRNOFFSpecError
  vdw  scale13=0.5           -> toolkit parse: SMIRNOFFSpecError
  vdw  scale15=0.5           -> toolkit parse: SMIRNOFFSpecError
  exclusion_policy=none      -> interchange export: UnsupportedExportError
  exclusion_policy=minimal   -> interchange export: UnsupportedExportError
  scale14=0.7/0.3 (in-model)  -> BUILT (accepted, applied consistently)

No silent-danger path: every out-of-model setting raises at parse or export.


In [11]:
import json
with open("results_01.json", "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print(f"\nWrote results_01.json with {len(RESULTS)} rows")
print("Notebook 01 complete: Interchange and openmmforcefields agree for every vsite type, "
      "charge method (OFFXML-specified and preset), net charge, ordering, and force layout tested.")


Wrote results_01.json with 30 rows
Notebook 01 complete: Interchange and openmmforcefields agree for every vsite type, charge method (OFFXML-specified and preset), net charge, ordering, and force layout tested.
